# Create Zhejiang NSF Awards

Creates awards for **Natural Science Foundation of Zhejiang Province** (OpenAlex F4320338464) from the province's
annual S&T-portal award-list attachments.

**Prerequisites:** run `scripts/local/zhejiang_nsf_to_s3.py` first.

**S3 location:** `s3a://openalex-ingest/awards/zhejiang_nsf/zhejiang_nsf_projects.parquet`

**Funder (Path A, F4320* -- in `openalex.common.funder`):**
- funder_id: 4320338464
- display_name: "Natural Science Foundation of Zhejiang Province"
- provenance: `zhejiang_nsf`
- priority: **403**

**Notes:** Native 立项编号 grant numbers; no per-project amounts published (Step 6.7 waiver).
PI names follow the NSFC/MOST convention: whole Chinese personal name in
`family_name`, `given_name` NULL. `id` uses the native grant number where
present, else a stable content-hash `row_key`.

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.zhejiang_nsf_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/zhejiang_nsf/zhejiang_nsf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.zhejiang_nsf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.zhejiang_nsf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.zhejiang_nsf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A -- must return exactly 1 row)

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder WHERE funder_id = 4320338464;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.zhejiang_nsf_awards
USING delta
AS
WITH
funder_src AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320338464
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':',
            COALESCE(LOWER(NULLIF(TRIM(g.funder_award_id), '')), g.row_key)))) % 9000000000 as id,

        g.display_name as display_name,
        CAST(NULL AS STRING) as description,

        f.funder_id,
        NULLIF(TRIM(g.funder_award_id), '') as funder_award_id,

        CAST(NULL AS DOUBLE) as amount,   -- no per-project amounts published (Step 6.7 waiver)
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,

        CASE
            WHEN g.funder_scheme LIKE '%杰出青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%优秀青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%青年%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%博士%' THEN 'fellowship'
            WHEN g.funder_scheme LIKE '%团队%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重大%' THEN 'research'
            WHEN g.funder_scheme LIKE '%重点%' THEN 'research'
            ELSE 'grant'
        END as funding_type,

        g.funder_scheme as funder_scheme,
        'zhejiang_nsf' as provenance,

        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        TRY_CAST(g.start_year AS INT) as start_year,
        TRY_CAST(g.end_year AS INT) as end_year,

        CASE
            WHEN (g.family_name IS NOT NULL AND TRIM(g.family_name) != '')
              OR (g.institution IS NOT NULL AND TRIM(g.institution) != '') THEN
                struct(
                    NULLIF(TRIM(g.given_name), '') as given_name,
                    NULLIF(TRIM(g.family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.institution), '') as name,
                        'China' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,

        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.zhejiang_nsf_raw g
    CROSS JOIN funder_src f
)
SELECT *,
    concat('https://api.openalex.org/works?filter=awards.id:G', id) as works_api_url
FROM awards_transformed;

## Step 3: Delete + insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'zhejiang_nsf' AND priority = 403;

-- Insert into openalex_awards_raw with priority 403 (direct funder ingest).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    403 as priority
FROM openalex.awards.zhejiang_nsf_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.zhejiang_nsf_awards;

In [ ]:
%sql
SELECT
  COUNT(*) as total,
  COUNT(display_name) as has_title,
  COUNT(funder_award_id) as has_grant_id,
  COUNT(amount) as has_amount,
  COUNT(lead_investigator.family_name) as has_pi,
  COUNT(lead_investigator.affiliation.name) as has_inst,
  ROUND(COUNT(amount)*100.0/COUNT(*),1) as pct_amount
FROM openalex.awards.zhejiang_nsf_awards;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) FROM openalex.awards.zhejiang_nsf_awards GROUP BY 1;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as n FROM openalex.awards.zhejiang_nsf_awards GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) FROM openalex.awards.zhejiang_nsf_awards GROUP BY 1 ORDER BY 1 DESC;

In [ ]:
%sql
SELECT lead_investigator.family_name, lead_investigator.affiliation.name, COUNT(*) n
FROM openalex.awards.zhejiang_nsf_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT provenance, priority, COUNT(*) n FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'zhejiang_nsf' GROUP BY 1,2;